# 学习日志 - Day2 (2026-03-19)

## 完成内容
- 学习了提示词工程基础（角色设定、少样本、思维链）
- 完成5个JSON输出练习
- 测试了temperature从0.2到2.0的效果

## 最佳实践
- JSON输出时，配合示例（Few-shot）效果最好
- 用temperature=0.3能让JSON格式更稳定
- 思维链确实能提高复杂问题的准确性

## 遇到的问题
- 第一次输出JSON时带了额外文字 → 解决方法：在system prompt里强调“只输出JSON，不要额外解释”

## 明日计划
- 学习文档加载和切分（为RAG做准备）

In [17]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

response = client.chat.completions.create(
    model="deepseek-chat",

    messages=[
    {"role": "system", "content": "你是一个待办事项助手。请始终输出纯净的 JSON 数组，不要包含任何其他文字、解释或 Markdown 代码块。"},
    {"role": "user", "content": "请列出今天的三条代办事项，包含时间、事项名称、优先级（高/中/低）。"}
    ],
    temperature=0.3,
    max_tokens=300
)

print(response.choices[0].message.content)
print("本次使用 token 数：",response.usage.total_tokens)
import json

content = response.choices[0].message.content
print("原始内容：", content)

try:
    data = json.loads(content)  # 尝试解析 JSON
    print("解析成功！数据是：", data)
    print("数据类型：", type(data))  # 可能是 list 或 dict
except json.JSONDecodeError as e:
    print("解析失败：", e)
    print("请检查 AI 返回的内容是否包含额外文字。")

[
  {
    "时间": "09:00",
    "事项名称": "完成项目报告",
    "优先级": "高"
  },
  {
    "时间": "14:00",
    "事项名称": "团队会议",
    "优先级": "中"
  },
  {
    "时间": "17:00",
    "事项名称": "回复客户邮件",
    "优先级": "低"
  }
]
本次使用 token 数： 147
原始内容： [
  {
    "时间": "09:00",
    "事项名称": "完成项目报告",
    "优先级": "高"
  },
  {
    "时间": "14:00",
    "事项名称": "团队会议",
    "优先级": "中"
  },
  {
    "时间": "17:00",
    "事项名称": "回复客户邮件",
    "优先级": "低"
  }
]
解析成功！数据是： [{'时间': '09:00', '事项名称': '完成项目报告', '优先级': '高'}, {'时间': '14:00', '事项名称': '团队会议', '优先级': '中'}, {'时间': '17:00', '事项名称': '回复客户邮件', '优先级': '低'}]
数据类型： <class 'list'>


In [20]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个中餐大师，严格按照要求输出"},
        {"role": "user", "content": """
请用JSON格式列出番茄炒蛋的步骤，格式如下：
[

  {"序号（1）""操作描述",  "预计时间"},

]

现在请输出番茄炒蛋步骤：
"""}
    ],
    temperature=2,
    max_tokens=300
)

print(response.choices[0].message.content)
print("本次使用 token 数：",response.usage.total_tokens)
import json

content = response.choices[0].message.content
print("原始内容：", content)

try:
    data = json.loads(content)  # 尝试解析 JSON
    print("解析成功！数据是：", data)
    print("数据类型：", type(data))  # 可能是 list 或 dict
except json.JSONDecodeError as e:
    print("解析失败：", e)
    print("请检查 AI 返回的内容是否包含额外文字。")

[
  {"序号": "1", "操作描述": "准备2个鸡蛋，1个番茄，适量葱、盐、糖等调料。鸡蛋打散搅匀，番茄洗净切块，葱切末备用", "预计时间": "5分钟"},
  {"序号": "2", "操作描述": "热锅倒油，油热后倒入蛋液，快速翻炒至蛋液凝固成块，盛出备用", "预计时间": "3分钟"},
  {"序号": "3", "操作描述": "原锅加热少许油，放入番茄块煸炒至出汁变软", "预计时间": "2分钟"},
  {"序号": "4", "操作描述": "将炒好的鸡蛋倒回锅中，与番茄一起翻炒均匀", "预计时间": "1分钟"},
  {"序号": "5", "操作描述": "加入适量的盐、少许糖调味，撒上葱花，翻炒均匀即可出锅装盘", "预计时间": "1分钟"}
]
本次使用 token 数： 267
原始内容： [
  {"序号": "1", "操作描述": "准备2个鸡蛋，1个番茄，适量葱、盐、糖等调料。鸡蛋打散搅匀，番茄洗净切块，葱切末备用", "预计时间": "5分钟"},
  {"序号": "2", "操作描述": "热锅倒油，油热后倒入蛋液，快速翻炒至蛋液凝固成块，盛出备用", "预计时间": "3分钟"},
  {"序号": "3", "操作描述": "原锅加热少许油，放入番茄块煸炒至出汁变软", "预计时间": "2分钟"},
  {"序号": "4", "操作描述": "将炒好的鸡蛋倒回锅中，与番茄一起翻炒均匀", "预计时间": "1分钟"},
  {"序号": "5", "操作描述": "加入适量的盐、少许糖调味，撒上葱花，翻炒均匀即可出锅装盘", "预计时间": "1分钟"}
]
解析成功！数据是： [{'序号': '1', '操作描述': '准备2个鸡蛋，1个番茄，适量葱、盐、糖等调料。鸡蛋打散搅匀，番茄洗净切块，葱切末备用', '预计时间': '5分钟'}, {'序号': '2', '操作描述': '热锅倒油，油热后倒入蛋液，快速翻炒至蛋液凝固成块，盛出备用', '预计时间': '3分钟'}, {'序号': '3', '操作描述': '原锅加热少许油，放入番茄块煸炒至出汁变软', '预计时间': '2分钟'}, {'序号': '4', '操作描述': '将炒好的鸡蛋倒回锅中，与番茄一起翻炒均匀', '预计时间'

In [23]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个重度电影爱好者，严格按照要求输出"},
        {"role": "user", "content": """

请推荐3部科幻电影，用JSON格式输出，包含电影名、导演、评分、一句话简介

"""}
    ],
    temperature=2,
    max_tokens=300
)

print(response.choices[0].message.content)
print("本次使用 token 数：",response.usage.total_tokens)
import json

content = response.choices[0].message.content
print("原始内容：", content)

try:
    data = json.loads(content)  # 尝试解析 JSON
    print("解析成功！数据是：", data)
    print("数据类型：", type(data))  # 可能是 list 或 dict
except json.JSONDecodeError as e:
    print("解析失败：", e)
    print("请检查 AI 返回的内容是否包含额外文字。")

{
    "电影": [
        {
            "电影名": "盗梦空间",
            "导演": "克里斯托弗·诺兰",
            "评分": 9.2,
            "一句话简介": "梦境与现实交织，盗梦专家进入他人意识"
        },
        {
            "电影名": "星际穿越",
            "导演": "克里斯托弗·诺兰",
            "评分": 9.0,
            "一句话简介": "末日之下，宇航员穿越虫洞拯救人类"
        },
        {
            "电影名": "攻壳机动队（1995）",
            "导演": "押井守",
            "评分": 8.9,
            "一句话简介": "人机结合的秘密军警在人工智能迷雾中寻求真相"
        }
    ]
}
本次使用 token 数： 205
原始内容： {
    "电影": [
        {
            "电影名": "盗梦空间",
            "导演": "克里斯托弗·诺兰",
            "评分": 9.2,
            "一句话简介": "梦境与现实交织，盗梦专家进入他人意识"
        },
        {
            "电影名": "星际穿越",
            "导演": "克里斯托弗·诺兰",
            "评分": 9.0,
            "一句话简介": "末日之下，宇航员穿越虫洞拯救人类"
        },
        {
            "电影名": "攻壳机动队（1995）",
            "导演": "押井守",
            "评分": 8.9,
            "一句话简介": "人机结合的秘密军警在人工智能迷雾中寻求真相"
        }
    ]
}
解析成功！数据是： {'电影': [{'电影名': '盗梦空间', '导演': '克里斯托弗·诺兰', '评分': 9.2, '一句话简介'

In [33]:

import json
import re
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个数学解题助手，请分步思考。让我们一步一步思考."},
        {"role": "user", "content": """
问题：小明有15个苹果，他给了小红3个，然后又买了5个，现在有多少个苹果？

请分步思考，逐步推理，最后给出答案。
让我们一步一步思考
"""}
    ],
    temperature=2,
    max_tokens=300
)


content = response.choices[0].message.content
print("原始内容：", content)
match = re.search(r'```json\s*([\s\S]*?)\s*```', content, re.DOTALL)
if match:
    json_str = match.group(1)  # 提取出 JSON 字符串
    try:
        data = json.loads(json_str)
        print("解析成功！数据：", data)
    except json.JSONDecodeError as e:
        print("JSON 解析失败：", e)
else:
    # 如果没有代码块，直接尝试解析
    try:
        data = json.loads(content)
        print("解析成功！数据：", data)
    except json.JSONDecodeError as e:
        print("JSON 解析失败：", e)

原始内容： 我们一步一步分析这个问题：  

**第一步：小明最初有多少个苹果？**  
小明最初有 15 个苹果。  

**第二步：他给了小红 3 个**  
此时剩下的苹果数 = \( 15 - 3 = 12 \) 个苹果。  

**第三步：他又买了 5 个苹果**  
现在的苹果数 = \( 12 + 5 = 17 \) 个苹果。  

**最终答案：** 小明现在有 **17** 个苹果。
本次使用 token 数： 166
原始内容： 我们一步一步分析这个问题：  

**第一步：小明最初有多少个苹果？**  
小明最初有 15 个苹果。  

**第二步：他给了小红 3 个**  
此时剩下的苹果数 = \( 15 - 3 = 12 \) 个苹果。  

**第三步：他又买了 5 个苹果**  
现在的苹果数 = \( 12 + 5 = 17 \) 个苹果。  

**最终答案：** 小明现在有 **17** 个苹果。
JSON 解析失败： Expecting value: line 1 column 1 (char 0)


In [34]:
import os
import json
import re
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个待办事项助手，请始终用JSON格式输出。"},
        {"role": "user", "content": "请列出今天的三条代办事项，包含时间、事项名称、优先级（高/中/低）。使用JSON格式。"}
    ],
    temperature=0.3,
    max_tokens=300
)

content = response.choices[0].message.content
print("原始内容：", content)

# 尝试去除可能的 Markdown 代码块
match = re.search(r'```json\s*([\s\S]*?)\s*```', content, re.DOTALL)
if match:
    json_str = match.group(1)
else:
    json_str = content  # 假设已经是纯净 JSON

try:
    data = json.loads(json_str)
    print("解析成功！数据：", data)
    print("第一条待办：", data["todo_list"][0])  # 示例访问数据
except json.JSONDecodeError as e:
    print("JSON 解析失败：", e)
    print("请检查 AI 返回的内容是否包含额外文字。")

原始内容： ```json
{
  "todo_list": [
    {
      "time": "09:00",
      "task_name": "完成项目进度报告",
      "priority": "高"
    },
    {
      "time": "14:30",
      "task_name": "团队周会",
      "priority": "中"
    },
    {
      "time": "16:00",
      "task_name": "整理办公桌文件",
      "priority": "低"
    }
  ]
}
```
解析成功！数据： {'todo_list': [{'time': '09:00', 'task_name': '完成项目进度报告', 'priority': '高'}, {'time': '14:30', 'task_name': '团队周会', 'priority': '中'}, {'time': '16:00', 'task_name': '整理办公桌文件', 'priority': '低'}]}
第一条待办： {'time': '09:00', 'task_name': '完成项目进度报告', 'priority': '高'}


In [38]:
# 测试不同temperature值对创意输出的影响
temperatures = [0.2, 0.7, 1.2, 2.0]  # DeepSeek的temperature范围是0-2

for temp in temperatures:
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content":"你是恋爱大师，效果非常有效"},
            {"role": "user", "content": "我喜欢一个女生，但是不知道怎么追她，好纠结，大二，她是我同班同学"}
        ],
        temperature=temp,
        max_tokens=300
    )
    print(f"temperature={temp}: {response.choices[0].message.content}")
    print("-" * 50)

temperature=0.2: 我能感受到你内心的悸动与忐忑。喜欢同班同学确实会让人既兴奋又紧张，毕竟你们每天都会见面，这种近距离的暗恋确实需要更细腻的处理方式。让我来帮你理清思路，找到最适合你的行动方案。

## 🔍 了解现状：你们目前的关系如何？

首先，我们需要评估一下你们现有的互动基础：
1. 你们平时有交流吗？比如课堂讨论、小组作业或日常打招呼？
2. 你对她了解多少？她的兴趣爱好、常去的地方、朋友圈子？
3. 她是否表现出对你的任何特别关注或友好？

## 🌱 建立自然连接：从同学到朋友的过渡

### 1️⃣ 创造自然接触机会
- **学习小组**：提议组建或加入她所在的学习小组，这是最自然的接触方式
- **课堂互动**：主动坐在她附近，讨论课程内容、借笔记、询问作业
- **共同兴趣**：了解她的兴趣爱好，寻找共同话题（音乐、电影、社团活动等）

### 2️⃣ 从线上到线下的渐进接触
- **社交媒体互动**：先点赞、评论她的朋友圈，逐渐开始私聊
- **学习相关话题**：从课程内容开始，慢慢扩展到生活话题
- **邀约技巧**：先提议小组活动（如“我们几个同学一起去图书馆复习吧”），再尝试一对一邀约

## 💡 具体行动步骤

### 第一阶段
--------------------------------------------------
temperature=0.7: 我能感受到你内心的悸动与忐忑，这种既甜蜜又纠结的心情是青春里最珍贵的体验。作为同班同学，你们已经有了天然的优势——共同的课程、活动和朋友圈，这些都是建立联系的绝佳机会。让我们一步步来，把这份心动转化为行动。

## 🌱 从同学到朋友的过渡策略

### 1️⃣ 创造自然接触的机会
* **学习小组邀请**：可以以课程项目或考试复习为由，邀请她一起学习。比如：“这周末我要去图书馆复习高数，听说你学得很好，可以一起吗？我有些问题想请教。”
* **课堂互动**：上课时选择坐在她附近，课间可以聊聊刚讲的内容或作业，从学术话题开始最自然。
* **班级活动参与**：积极参与班级组织的活动，如团建、运动会等，在这些场合中寻找与她互动的机会。

### 2️⃣ 建立舒适的朋友关系
* **从共同兴趣入手**：观察她的朋友圈、日常聊天中透露的兴趣爱好，找到共同话题。比如她喜欢某位歌

In [39]:
!git remote add origin https://github.com/naizzzkl/ai-agent-learning.git

In [41]:
!git branch -M main          # 确保本地分支名为 main
!git push -u origin main     # 推送并关联上游

fatal: too many arguments for a rename operation
error: src refspec # does not match any
error: src refspec 鎺ㄩ�佸苟鍏宠仈涓婃父 does not match any
error: failed to push some refs to 'https://github.com/naizzzkl/ai-agent-learning.git'


In [43]:
!git --version

git version 2.53.0.windows.2


In [44]:
!git config --global user.name "naizzzkl"
!git config --global user.email "naizzzkl@gmail.com"

In [45]:
!git init
!git add .
!git commit -m "first commit: 成功调用 DeepSeek API"

Reinitialized existing Git repository in C:/Users/naizz/ai_first_call/.git/


[main 16dfeb2] first commit: 鎴愬姛璋冪敤 DeepSeek API
 3 files changed, 776 insertions(+), 6 deletions(-)
 create mode 100644 Untitled1.ipynb
 create mode 100644 log.md


In [46]:
!git init

Reinitialized existing Git repository in C:/Users/naizz/ai_first_call/.git/


In [47]:
!git add .

In [48]:
!git commit -m "first commit: 成功调用 DeepSeek API"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [51]:
!git push

fatal: unable to access 'https://github.com/naizzzkl/ai-agent-learning.git/': Failed to connect to github.com port 443 after 21101 ms: Could not connect to server


In [1]:
pip list | findstr langchain

Note: you may need to restart the kernel to use updated packages.
